In [1]:
import os
import sys

AESARA_FLAGS="blas__ldflags=,cxx="
from aesara_theano_fallback import aesara as theano

KeyboardInterrupt: 

In [1]:
pip show aesara aesara-theano-fallback theano

Name: aesara
Version: 2.8.12
Summary: A library for defining, optimizing, and efficiently evaluating mathematical expressions involving multi-dimensional arrays.
Home-page: https://github.com/aesara-devs/aesara
Author: 
Author-email: aesara-devs <aesara.devs@gmail.com>
License: BSD-3-Clause
Location: /Users/ssagear/miniforge3-x86_64/envs/alderaan_4/lib/python3.9/site-packages
Requires: cons, etuples, filelock, logical-unification, minikanren, numpy, scipy, setuptools, typing-extensions
Required-by: 
---
Name: aesara-theano-fallback
Version: 0.1.0
Summary: Striving towards backwards compatibility with the Theano -> Aesara transition
Home-page: https://github.com/exoplanet-dev/aesara-theano-fallback
Author: Dan Foreman-Mackey, Rodrigo Luger
Author-email: foreman.mackey@gmail.com
License: MIT
Location: /Users/ssagear/miniforge3-x86_64/envs/alderaan_4/lib/python3.9/site-packages
Requires: 
Required-by: exoplanet, pymc3-ext
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
import sys
# os.environ["AESARA_FLAGS"] = "device=cpu,floatX=float64,optimizer=None"
# os.environ["AESARA_FLAGS"] = "compiledir_lock=0"

os.environ["AESARA_FLAGS"] = "blas__ldflags="
# from aesara_theano_fallback import aesara as theano
import argparse
from astropy.units import UnitsWarning
from astropy.stats import mad_std
from celerite2.backprop import LinAlgError
from configparser import ConfigParser
from datetime import datetime
import gc
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import shutil
from timeit import default_timer as timer
import warnings

In [2]:
from alderaan.constants import *
from alderaan.ephemeris import Ephemeris
from alderaan.litecurve import LiteCurve
from alderaan.litecurve import KeplerLiteCurve
from alderaan.planet import Planet
# from alderaan.modules.detrend import GaussianProcessDetrender
# from alderaan.modules.omc import OMC
# from alderaan.modules.transit_model import ShapeTransitModel, TTimeTransitModel
from alderaan.modules.quality_control import QualityControl
# from alderaan.modules.quicklook import plot_litecurve, plot_omc, dynesty_cornerplot, dynesty_runplot, dynesty_traceplot
from alderaan.utils.io import resolve_config_path, parse_koi_catalog, parse_holczer16_catalog, copy_input_target_catalog


In [3]:
def initialize_pipeline():
    # flush buffer
    sys.stdout.flush()
    sys.stderr.flush()

    # filter warnings
    warnings.simplefilter('always', UserWarning)
    warnings.filterwarnings(
        action='ignore', category=UnitsWarning, module='astropy'
    )

    # start timer
    global_start_time = timer()

    return global_start_time


In [4]:
def cleanup():
    sys.stdout.flush()
    sys.stderr.flush()
    plt.close('all')
    gc.collect()


In [5]:
global_start_time = initialize_pipeline()

print("")
print("+" * shutil.get_terminal_size().columns)
print("ALDERAAN Pipeline")
print(f"Initialized {datetime.now().strftime('%d-%b-%Y at %H:%M:%S')}")
print("+" * shutil.get_terminal_size().columns)
print("")


++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
ALDERAAN Pipeline
Initialized 08-May-2026 at 13:56:18
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++



In [7]:
import argparse
args = argparse.Namespace(mission='Kepler', target='K00137', config='/Users/ssagear/Documents/Research/alderaan/alderaan/configs/default_config.cfg')

config = ConfigParser()
config.read(args.config)

['/Users/ssagear/Documents/Research/alderaan/alderaan/configs/default_config.cfg']

In [8]:
# alderaan_base_path = Path(__file__).resolve().parents[2]
# for key, value in config["PATHS"].items():
#     config['PATHS'][key] = resolve_config_path(config['PATHS'][key], alderaan_base_path)

In [9]:
mission = args.mission
target = args.target
run_id = config['RUN']['run_id']

data_dir =  '/Users/ssagear/Documents/Research/alderaan/alderaan/examples/data/MAST_downloads/'
outputs_dir = 'exploratory_outputs'
catalog_dir = 'exploratory_outputs'

catalog_csv = os.path.join(catalog_dir, str(config['ARGS']['catalog_csv']))

print("")
print(f"   MISSION : {mission}")
print(f"   TARGET  : {target}")
print(f"   RUN ID  : {run_id}")
print("")
print(f"   Data directory    : {data_dir}")
print(f"   Config file       : {args.config}")
print(f"   Input catalog     : {os.path.basename(catalog_csv)}")
print("")
# print(f"   theano cache : {theano.config.compiledir}")
print("")


   MISSION : Kepler
   TARGET  : K00137
   RUN ID  : develop

   Data directory    : /Users/ssagear/Documents/Research/alderaan/alderaan/examples/data/MAST_downloads/
   Config file       : /Users/ssagear/Documents/Research/alderaan/alderaan/configs/default_config.cfg
   Input catalog     : kepler_dr25_gaia_dr2_crossmatch.csv




In [10]:
# build directory structure
os.makedirs(outputs_dir, exist_ok=True)

results_dir = os.path.join(outputs_dir, 'results', run_id, target)
os.makedirs(results_dir, exist_ok=True)

quicklook_dir = os.path.join(outputs_dir, 'quicklook', run_id, target)
os.makedirs(quicklook_dir, exist_ok=True)

# copy input catalog into results directory
catalog_csv_copy = os.path.join(outputs_dir, 'results', run_id, f'{run_id}.csv')
# copy_input_target_catalog(catalog_csv, catalog_csv_copy)


In [14]:
# ######### #
# I/O Block #
# ######### #

print('\n\nI/O BLOCK\n')

# load KOI catalog
catalog = parse_koi_catalog(catalog_csv, target)


assert np.all(np.diff(catalog.period) > 0), "Planets should be ordered by ascending period"

NPL = int(catalog.npl[0])
koi_id = catalog.koi_id[0]
kic_id = int(catalog.kic_id[0])

# load lightcurves
#litecurve_master = LiteCurve(data_dir, kic_id, 'long cadence', data_source='Kepler PDCSAP')
# litecurve_master = LiteCurve().load_kepler_pdcsap(data_dir, kic_id, 'long cadence', visits=2)
litecurve_master = KeplerLiteCurve().load_kepler_pdcsap(data_dir, kic_id, 'long cadence', quarters=2)





I/O BLOCK



In [16]:
litecurve_master

In [19]:
t_min = litecurve_master.time.min()
t_max = litecurve_master.time.max()
if t_min < 0:
    raise ValueError("Lightcurve has negative timestamps...this will cause problems")

In [23]:
# split litecurves by visit
litecurves = litecurve_master.split_visits()

for j, litecurve in enumerate(litecurves):
    assert len(np.unique(litecurve.visit)) == 1, "expected one quarter per litecurve"
    assert len(np.unique(litecurve.obsmode)) == 1, "expected one obsmode per litecurve"

print(f"{len(litecurves)} litecurves loaded for {target}")



1 litecurves loaded for K00137


In [25]:
# initialize planets (catch no ephemeris warning)
with warnings.catch_warnings(record=True) as catch:
    warnings.simplefilter('always', category=UserWarning)
    planets = [None]*NPL
    for n in range(NPL):
        planets[n] = Planet(catalog, target, n)


print(f"\n{NPL} planets loaded for {target}")
print([np.round(p.period,6) for p in planets])


3 planets loaded for K00137
[3.504691, 7.641568, 14.858909]


In [27]:
# update planet ephemerides
for n, p in enumerate(planets):
    if p.ephemeris is None:
        _ephemeris = Ephemeris(period=p.period, epoch=p.epoch, t_min=t_min, t_max=t_max)
        planets[n] = p.update_ephemeris(_ephemeris)

# load Holczer+2016 catalog
filepath = os.path.join(catalog_dir, 'holczer_2016_kepler_ttvs.txt')
holczer_ephemerides = parse_holczer16_catalog(filepath, koi_id, NPL)

print(f"\n{len(holczer_ephemerides)} ephemerides found in Holczer+2016")



3 ephemerides found in Holczer+2016


In [28]:
# match Holczer ephemerides to Planets
count = 0

for n, p in enumerate(planets):
    for ephem in holczer_ephemerides:
        match = np.isclose(ephem.period, p.period, rtol=0.01, atol=p.duration)

        if match:
            print(f"  Planet {n} : {p.period:.6f} --> {ephem.period:.6f}")
            planets[n] = p.update_ephemeris(ephem)
            count += 1

print(f"{count} matching ephemerides found ({len(holczer_ephemerides)} expected)")



  Planet 0 : 3.504691 --> 3.504722
  Planet 1 : 7.641568 --> 7.641562
  Planet 2 : 14.858909 --> 14.858844
3 matching ephemerides found (3 expected)


In [29]:
# quicklook litecurve
filepath = os.path.join(quicklook_dir, f"{target}_litecurve_raw.png")
_ = plot_litecurve(litecurve_master, target, planets, filepath)

# end-of-block cleanup
# cleanup()

NameError: name 'plot_litecurve' is not defined